# Integrating systematic weights into the binned w(θ) pipeline
Drop-in cells for `w_theta_binned_refactored.ipynb`, using `sys_weights.py`.

**Estimator convention fixed here (important):** weights are applied to the **data only**,
randoms stay unweighted. Weighting both data and randoms with the same map (as the earlier
refactor did) makes the correction cancel pixel-by-pixel in Landy–Szalay; weighting the *randoms*
with `1/(1+δ_pred)` (as the original notebook did) applies the correction with the **wrong sign**
and amplifies contamination instead of removing it. The two valid conventions are:
data × `1/(1+δ_pred)` with plain randoms — used here — or, equivalently,
randoms × `(1+δ_pred)` with plain data. Pick one and never mix.

In [4]:
import fitsio
path = 'table_match_final.fits'
fitsio.FITS(path)[1].read()

array([(165253, '0027m300',  965, 'SER',   2.62843216, -29.91925625, [ 612854.25,  715293.5 ,  777498.56,  744897.7 ,  778189.3 ], 0.01254878,  29.589272,  69.85892,  120.78643,  106.40177,  57.43534, -102.60618 ,  -840.277  , 177.3959   , 84.66146   , 14.212937  , 1.5068868 , 0.47010678, 0.00085539, 1.2533548e-05, 0.96353453, 0.97528774, 0.9861009 , 0.99787563, 0.99869484, 0.99972147, 0.99989486, 4, 3, 3, 8.2563730e-03, 3.8269241e-03, 2.2503377e-03, 0.1046411 , 0.00746801, 0.00822243, 0.9908769 , 0.9793203 , 0.952676  , 1.4679358, 1.1513642, 0.93300533, 1731.0267 , 1207.1104  , 116.29737 ,  1.439051 , 0.05670692,  0.4707824 , 10.633193 , 25.104483, 43.40578 , 19.594368,    0, 5.365548 ,  0.8594418, 18.781832, 17.862278, 17.279758, 17.430319, 19.89301 , 18.973454, 18.390934, 18.822165, 17.889446, 17.294954 , 0.11925707, 0.07404   , -99., ' ', False, nan, 'NONE'),
       (165253, '0027m300', 8018, 'SER',   2.86354092, -29.98558342, [1724862.2 , 2687997.2 , 2944690.  , 2751313.5 , 294516

In [1]:
from sys_weights import SysWeights

WEIGHT_MODE = 'global'      # 'none' | 'global' | 'per_bin'
SYS_DIR     = 'imaging_systematics_maps_new/data'

# NOTE: one dec split everywhere. The regression notebook used 32.3 while the
# clustering pipeline used 32.375 — galaxies in that strip were weighted with
# the wrong region's map. DEC_SPLIT from the w(θ) notebook is the single truth.
SW = {p: SysWeights(part=p,
                    mask_path=f'mask_psf_{p}_nside_64.fits',
                    sys_dir=SYS_DIR,
                    rand_counts_path=f'{SYS_DIR}/{p}_counts_randoms_nside_64.fits')
      for p in ('north', 'south')}

def _part_of(dec):
    return np.where(np.asarray(dec) >= DEC_SPLIT, 'north', 'south')

/user/animesh.sah/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


## Global fit (once per region)
χ²/dof before → after is the sanity number: it should drop toward ~1. If χ²_after ≪ expected,
you are overcorrecting (absorbing real clustering into the weights).

In [2]:
GLOBAL_FIT = {}
if WEIGHT_MODE != 'none':
    for p, sw in SW.items():
        reg = (DEC_all >= DEC_SPLIT) if p == 'north' else (DEC_all < DEC_SPLIT)
        GLOBAL_FIT[p] = sw.fit(RA_all[reg], DEC_all[reg])
        f = GLOBAL_FIT[p]
        print(f"{p}: chi2/dof  {f['chi2_before']/f['ndof']:.2f} -> "
              f"{f['chi2_after']/f['ndof']:.2f}")

NameError: name 'DEC_all' is not defined

## Replacement `get_weights` for the binned notebook
Replaces the map-file-based version. `bin_fit` overrides the global fit when running per-bin.
Galaxies on pixels excluded from the fit footprint (outlier pixels) return NaN — the driver below
**drops them from the measurement** instead of silently giving them weight 1, so data and the
correction share one footprint. If you keep the randoms as they are, apply the same
`fit_pixel_mask` cut to them once (cell below) so D and R windows match.

In [ ]:
def get_weights(ra, dec, region='auto', bin_fits=None):
    if WEIGHT_MODE == 'none':
        return None
    dec = np.asarray(dec)
    parts = _part_of(dec) if region == 'auto' else np.full(len(dec),
             'north' if region == 'N' else 'south')
    w = np.empty(len(dec))
    for p in ('north', 'south'):
        m = parts == p
        if not m.any():
            continue
        fit = (bin_fits or GLOBAL_FIT).get(p, GLOBAL_FIT[p])
        w[m] = SW[p].galaxy_weights(fit, np.asarray(ra)[m], dec[m])
    return w   # NaN on excluded pixels — caller must drop those rows

def apply_footprint_to_randoms(ra_r, dec_r):
    """Cut randoms to the union of the N/S fit footprints (run once)."""
    keep = np.zeros(len(ra_r), dtype=bool)
    parts = _part_of(dec_r)
    for p in ('north', 'south'):
        m = parts == p
        pix = hp.ang2pix(64, np.radians(90 - np.asarray(dec_r)[m]),
                         np.radians(np.asarray(ra_r)[m]))
        keep[m] = GLOBAL_FIT[p]['fit_pixel_mask'][pix]
    return keep

## Driver hook: global vs per-bin weights
In `run_property_group`, weights for each bin come from either the global fit or a refit on the
bin's own galaxies. Per-bin refits reuse the **same footprint, outlier cuts and design matrix**
(all precomputed in `SysWeights`), so a refit is just one WLS solve — cheap.

Rule of thumb before refitting everything: run `null_test` per bin with the **global** weights.
Only bins showing residual trends (χ²_after/dof ≫ 1 against flat for some systematic) need their
own fit.

In [ ]:
def weights_for_bin(ra_bin, dec_bin, mode=None):
    mode = mode or WEIGHT_MODE
    if mode == 'none':
        return None, None
    if mode == 'global':
        return get_weights(ra_bin, dec_bin), None
    # per_bin: refit on this bin's galaxies, each region separately
    parts = _part_of(dec_bin)
    bin_fits = {}
    for p in ('north', 'south'):
        m = parts == p
        if m.sum() > 2000:            # too few galaxies -> noisy fit, keep global
            bin_fits[p] = SW[p].fit(np.asarray(ra_bin)[m], np.asarray(dec_bin)[m])
    w = get_weights(ra_bin, dec_bin, bin_fits=bin_fits)
    return w, bin_fits

# inside make_bins_full, replace the weight line with:
#     w, _ = weights_for_bin(ra[mask], dec[mask])
#     keep = np.isfinite(w) if w is not None else np.ones(mask.sum(), bool)
#     bins.append((label, ra[mask][keep], dec[mask][keep],
#                  w[keep] if w is not None else None, cat_rand, rr))
# and in build_cat_rand_and_rr calls: pass wgt_r=None (randoms unweighted),
# after cutting them once with apply_footprint_to_randoms().

## Per-bin null test with global weights (the decision diagnostic)

In [ ]:
# Example: does the faintest MAG_R bin need its own weights?
# lbl = list(MASKS['MAG_R'])[-1]                      # faintest bin
# m   = MASKS['MAG_R'][lbl]
# for p in ('north','south'):
#     reg = (DEC_all >= DEC_SPLIT) if p=='north' else (DEC_all < DEC_SPLIT)
#     nt = SW[p].null_test(GLOBAL_FIT[p], RA_all[m & reg], DEC_all[m & reg])
#     for s, d in nt.items():
#         chi2_flat = d['after'][2] / len(d['centers'])
#         flag = '  <-- refit this bin' if chi2_flat > 2 else ''
#         print(f"{p} {s:>28}: chi2/bin after = {chi2_flat:5.2f}{flag}")